In [0]:
%load_ext autoreload
%autoreload 2

# Treinamento do modelo:

Este notebook tem como objetivo treinar um modelo de Rede Neural Convolucional (CNN) e conduzir experimentos utilizando o MLflow, com a finalidade de identificar e selecionar o modelo com melhor desempenho.

Serão testados:

- **SimpleCNN:**

    Modelo criado do zero para o problema em questão.

- **ResNet-18:**

    Rede neural convolucional profunda pré treinada com 18 camadas.

- **EfficientNet-b0:**

    Faz parte da família EfficientNet desenvolvida pelo Google, sendo b0 um modelo leve e de alta eficiência.

- **DenseNet-121:**

    Rede neural convolucional profunda, famosa pela arquitetura densamente conectada. Opera melhor com bases de dados maiores, será testada apenas como experimento.

Para efeitos de comparação, em cada run serão logados os seguintes parametros:

- Nome do modelo;
- Taxa de aprendizado;
- Freeze Backbone;
- Épocas;
- Classes;
- Early stopping (caso ocorra)

Para avaliação do desempenho serão logadas as métricas train_loss, train_acc, val_loss, val_acc.

Para cada modelo, serão treinadas duas versões distintas. A primeira utilizará o learning rate definido a partir do lr_finder, enquanto a segunda será treinada aplicando a estratégia One Cycle Policy. O objetivo é comparar o desempenho entre as duas abordagens e selecionar a configuração que apresentar os melhores resultados.

In [0]:
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

from PIL import Image
import io
import tempfile
import os

import torch
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from src.train import find_best_lr, run_training
from src.dataset.milho_dataset import get_dataloaders_milho
from src.evaluate import get_predictions
from src.models.milho_models import get_model


## Modelo criado do zero:

#### Learning Rate Finder CNN:

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "SimpleCNN"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

In [0]:
run_id = "085eca5d71904b6dbf0c9339df7b3d9f"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr = float(params.get("suggested_lr"))
suggested_lr

O melhor Leanring Rate encotrado para o modelo foi de 0.0014849682622544637.

#### CNN Learning Rate fixo:

Nesta etapa será treinado o modelo customizado utilizando o learning rate encontrado.

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "SimpleCNN"
learning_rate = suggested_lr
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_CNN, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_CNN(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_CNN,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

Durante o treinamento o Early stopping foi acionado, já que não houve melhora na acurácia da validação.
Das 30 épocas programadas para o treinamento, foram concluídas 21.

## Matriz confusão CNN LR fixo

Para entender melhor o poder de diferenciação do modelo, será utilizada a matriz confusão. Dessa maneira é possível analisar quais classes o modelo identifica com mais facilidade.

In [0]:
run_id_CNN = "6d60be5f39024202b9fb0863811c8632"

with mlflow.start_run(run_id=run_id_CNN):

    val_labels, val_preds = get_predictions(model_CNN, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_Simple_CNN.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_Simple_CNN.png")

## CNN 1cycle policy

Treinamento do modelo utilizando o 1Cycle Policy, onde o learning rate varia a cada época de treinamento.

In [0]:
run_id = "085eca5d71904b6dbf0c9339df7b3d9f"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr = float(params.get("suggested_lr"))

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "SimpleCNN"
learning_rate = suggested_lr
epochs = 25
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_CNN_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_CNN_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_CNN_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão CNN 1cycle policy

In [0]:
run_id_CNN_1cycle = "39efe20dccaf4baf9c33e4e128c93067"

with mlflow.start_run(run_id=run_id_CNN_1cycle):

    val_labels, val_preds = get_predictions(model_CNN_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_Simple_CNN_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_Simple_CNN_1cycle.png")

## LR finder Resnet18

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "resnet18"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

## Resnet18 LR fixo

In [0]:
run_id = "97cb6892487643f28c4c42bcf3b198be"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_resnet = float(params.get("suggested_lr")) / 10
suggested_lr_resnet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "resnet18"
learning_rate = suggested_lr_resnet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_resnet18, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_resnet18(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_resnet18,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão Resnet18 LR fixo

In [0]:
run_id_resnet18 = "ecfe31e2a87c42cab69b824c7e7b4fa2"

with mlflow.start_run(run_id=run_id_resnet18):

    val_labels, val_preds = get_predictions(model_resnet18, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_resnet18.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_resnet18.png")


## Resnet18 1cycle policy

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "resnet18"
learning_rate = suggested_lr_resnet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_resnet18_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_resnet18_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_resnet18_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

In [0]:
run_id_resnet18_1cycle = "67254b3ec1ec4a0b97a5835eafaee785"
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
model_resnet18_1cycle = mlflow.pytorch.load_model(f"runs:/{run_id_resnet18_1cycle}/model")
_, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with mlflow.start_run(run_id=run_id_resnet18_1cycle):

    val_labels, val_preds = get_predictions(model_resnet18_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_resnet18_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_resnet18_1cycle.png")

## LR finder efficientnet b0

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "efficientnet_b0"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

## Efficientnet_b0 LR fixo

In [0]:
run_id = "9d6edde8c6694ea5a888ab70dcf04695"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_effnet = float(params.get("suggested_lr"))
suggested_lr_effnet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "efficientnet_b0"
learning_rate = suggested_lr_effnet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_effnetb0, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_effnetb0(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_effnetb0,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão Efficientnet_b0 LR fixo

In [0]:
run_id_effnet = "a6af598f96c243e9b60e29937eb6aec5"

with mlflow.start_run(run_id=run_id_effnet):

    val_labels, val_preds = get_predictions(model_effnetb0, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_efnet.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_efnet.png")

## Efficientnet_b0 1cycle_policy

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "efficientnet_b0"
learning_rate = suggested_lr_effnet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_effnetb0_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_effnetb0_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_effnetb0_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão Efficientnet 1cycle

In [0]:
run_id_effnet_1cycle = "e3f855c9716c47dab3f623a344e6ae05"

with mlflow.start_run(run_id=run_id_effnet_1cycle):

    val_labels, val_preds = get_predictions(model_effnetb0_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_efnet_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_efnet_1cycle.png")

## LR finder Densenet121

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "densenet121"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

## Densenet121 LR fixo

In [0]:
run_id = "9375f9c7b8284e458de6969fabda6a0e"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_densenet = float(params.get("suggested_lr"))
suggested_lr_densenet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "densenet121"
learning_rate = suggested_lr_densenet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_densenet, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_densenet(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_densenet,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão densenet121 lr fixo:

In [0]:
run_id_densenet = "aafaa31d71f5460d8cb78214cd97f94d"

with mlflow.start_run(run_id=run_id_densenet):

    val_labels, val_preds = get_predictions(model_densenet, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_dense.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_dense.png")

## Densenet121 1cycle policy

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "densenet121"
learning_rate = suggested_lr_densenet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_densenet_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_densenet_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_densenet_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão densenet121 1cycle policy

In [0]:
run_id_densenet_1cycle = "5af183c6045c4698816dd0f6d711bc4e"
artifact_path = "model"
model_uri = f"runs:/{run_id_densenet_1cycle}/{artifact_path}"
model_densenet_1cycle = mlflow.pytorch.load_model(model_uri)

data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
_ , val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

with mlflow.start_run(run_id=run_id_densenet_1cycle):


    val_labels, val_preds = get_predictions(model_densenet_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_dense_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_dense_1cycle.png")

In [0]:
experiment_name = "/Shared/Milho_Doencas"

experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_compare = runs[[
    "run_id",
    "params.model_name",
    "params.learning_rate",
    "params.use_onecycle",
    "params.epochs",
    "metrics.train_acc",
    "metrics.val_acc",
    "metrics.train_loss",
    "metrics.val_loss"
]].copy()

df_compare.columns = [
    "run_id",
    "model",
    "lr",
    "onecycle",
    "epochs",
    "train_acc",
    "val_acc",
    "train_loss",
    "val_loss"
]

df_compare["score"] = (
    df_compare["val_acc"] - 0.1 * df_compare["val_loss"]
)

df_compare = df_compare.sort_values(
    by="score",
    ascending=False
)

df_compare.head(10)

### 🎯 Seleção do Modelo

A escolha do modelo final foi realizada com base em uma análise quantitativa e qualitativa dos experimentos registrados no MLflow.

#### 📊 Métricas consideradas:

- **Acurácia de validação (val_acc)**: utilizada como principal indicador de desempenho geral
- **Loss de validação (val_loss)**: utilizada para avaliar a confiança e capacidade de generalização do modelo
- **Matriz de confusão**: utilizada para análise qualitativa dos erros por classe

#### 🧠 Estratégia de seleção:

Os modelos foram ranqueados utilizando um score combinado:

score = val_acc - 0.1 * val_loss

Essa abordagem permite:

- Priorizar modelos com alta acurácia
- Penalizar modelos com baixa confiança (loss elevada)

#### 🥇 Critério final:

1. Maior score combinado  
2. Menor loss de validação (em caso de empate)  
3. Análise da matriz de confusão  

#### 🔍 Análise qualitativa:

Após o ranqueamento, os melhores modelos foram avaliados por meio da matriz de confusão para identificar:

- Classes com maior taxa de erro  
- Possíveis confusões entre doenças similares  
- Comportamentos indesejados do modelo  

O modelo final foi selecionado com base no melhor equilíbrio entre desempenho quantitativo e robustez qualitativa.

In [0]:
best_run = df_compare.iloc[0]

print("🏆 Melhor modelo encontrado:")
print(best_run)

## 📦 Re-empacotamento do Modelo com MLflow PyFunc

Para resolver o problema de dependências da pasta `src` ao carregar o modelo em outros ambientes (como a API), vamos re-empacotar o melhor modelo usando **MLflow PyFunc**.

### Por que PyFunc?

- **Portabilidade**: Empacota modelo + código customizado + dependências em um único artefato
- **Independência**: Não requer a pasta `src` no ambiente de destino
- **Compatibilidade**: Funciona com qualquer framework (PyTorch, TensorFlow, Scikit-learn, etc.)

### Estratégia:

1. Criar uma classe wrapper `MilhoDoencasPyFunc` que herda de `mlflow.pyfunc.PythonModel`
2. Incluir todo o código necessário dentro da classe (sem imports de `src`)
3. Re-logar o modelo melhor ranqueado usando `mlflow.pyfunc.log_model()`
4. Registrar no Unity Catalog com alias `@production`

In [0]:


class MilhoDoencasPyFunc(mlflow.pyfunc.PythonModel):
    """
    Wrapper PyFunc para o modelo de classificação de doenças do milho.
    Encapsula toda a lógica de preprocessing e inferência.
    """
    
    def load_context(self, context):
        """Carrega o modelo PyTorch do artefato."""
        import torch
        
        # Carrega o modelo PyTorch salvo (com weights_only=False para modelos customizados)
        model_path = context.artifacts["pytorch_model"]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = torch.load(model_path, map_location=self.device, weights_only=False)
        self.model.eval()
        
        # Define as classes
        self.classes = ["Blight", "Common_Rust", "Gray_Leaf_Spot", "Healthy"]
        
        # Define as transformações de preprocessing
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
    
    def predict(self, context, model_input):
        """Executa predição em imagens.
        
        Args:
            model_input: pode ser um array numpy (batch de imagens) ou
                        um dicionário com chave 'image_bytes' contendo bytes da imagem
        
        Returns:
            Lista de dicionários com 'prediction' e 'confidence'
        """
        import torch
        import torch.nn.functional as F
        import numpy as np
        
        # Se input é um dicionário com bytes da imagem (para API)
        if isinstance(model_input, dict) and 'image_bytes' in model_input:
            image_bytes = model_input['image_bytes']
            image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(image_tensor)
                probs = F.softmax(outputs, dim=1)
                confidence, pred = torch.max(probs, dim=1)
            
            return [{
                "prediction": self.classes[pred.item()],
                "confidence": float(confidence.item())
            }]
        
        # Se input é numpy array (batch de tensores preprocessados)
        else:
            tensor_input = torch.from_numpy(model_input).float().to(self.device)
            
            with torch.no_grad():
                outputs = self.model(tensor_input)
                probs = F.softmax(outputs, dim=1)
                confidence, pred = torch.max(probs, dim=1)
            
            results = []
            for i in range(len(pred)):
                results.append({
                    "prediction": self.classes[pred[i].item()],
                    "confidence": float(confidence[i].item())
                })
            
            return results

In [0]:
best_run_id = best_run['run_id']
print(f"Re-empacotando modelo do run: {best_run_id}")
print(f"Modelo: {best_run['model']}")
print(f"Val Accuracy: {best_run['val_acc']:.4f}")
print(f"Val Loss: {best_run['val_loss']:.4f}")

local_model_path = mlflow.artifacts.download_artifacts(f"runs:/{best_run_id}/model")
model_file = os.path.join(local_model_path, "data/model.pth")

original_model = torch.load(model_file, map_location='cpu', weights_only=False)


with tempfile.TemporaryDirectory() as tmpdir:
    model_path = os.path.join(tmpdir, "model.pth")
    torch.save(original_model, model_path)
    
    artifacts = {
        "pytorch_model": model_path
    }
    
    data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
    train_loader, _, _, _ = get_dataloaders_milho(data_dir=data_dir)
    example_input, _ = next(iter(train_loader))
    
    pyfunc_model = MilhoDoencasPyFunc()
    
    with mlflow.start_run(run_name=f"{best_run['model']}_pyfunc") as run:
        
        mlflow.log_param("original_run_id", best_run_id)
        mlflow.log_param("model_name", best_run['model'])
        mlflow.log_param("packaging_method", "pyfunc")
        mlflow.log_param("val_acc", best_run['val_acc'])
        mlflow.log_param("val_loss", best_run['val_loss'])
        
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=pyfunc_model,
            artifacts=artifacts,
            input_example=example_input[:1].numpy(),
            pip_requirements=[
                "torch",
                "torchvision",
                "pillow",
                "numpy"
            ]
        )